# Core Database Setup - UDA-Hub

This notebook sets up the UDA-Hub core database with accounts, users, tickets, knowledge base, and an initial ticket.

In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

## Core Database

### Init DB

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

Removed existing data/core/udahub.db
2026-07-02 18:16:21,466 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-02 18:16:21,466 INFO sqlalchemy.engine.Engine COMMIT
Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

### Account

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

### Knowledge Base

In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            cultpass_articles.append(json.loads(line))

print(f"Loaded {len(cultpass_articles)} articles")
for i, article in enumerate(cultpass_articles):
    print(f"  {i+1}. {article['title']}")

Loaded 14 articles
  1. How to Reserve a Spot for an Event
  2. What's Included in a CultPass Subscription
  3. How to Cancel or Pause a Subscription
  4. How to Handle Login Issues?
  5. How to Update Your Payment Method
  6. Understanding Your Monthly Billing Cycle
  7. How to Request a Refund
  8. Troubleshooting App Crashes and Errors
  9. How to Update Your Profile Information
  10. Premium Experience Upgrade Guide
  11. How to Transfer a Reservation to Another Person
  12. What to Do If an Experience Is Sold Out
  13. CultPass Referral Program
  14. Contacting Human Support and Escalation Policy


In [8]:
# Validate: must have at least 14 articles
assert len(cultpass_articles) >= 14, f"Expected at least 14 articles, got {len(cultpass_articles)}"

In [9]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb)
    print(f"Inserted {len(kb)} knowledge articles")

Inserted 14 knowledge articles


### Sample Ticket

In [10]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [11]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [12]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )

    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])
    print(f"Created ticket: {ticket.ticket_id}")

Created ticket: 92139507-8ca4-451c-874f-e5b3d3c50a07


## Tests

In [13]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(f"Account: {account}")
    print(f"Knowledge articles: {len(account.knowledge_articles)}")
    for article in account.knowledge_articles:
        print(f"  - {article.title}")

Account: <Account(account_id='cultpass', account_name='CultPass Card')>
Knowledge articles: 14
  - How to Reserve a Spot for an Event
  - What's Included in a CultPass Subscription
  - How to Cancel or Pause a Subscription
  - How to Handle Login Issues?
  - How to Update Your Payment Method
  - Understanding Your Monthly Billing Cycle
  - How to Request a Refund
  - Troubleshooting App Crashes and Errors
  - How to Update Your Profile Information
  - Premium Experience Upgrade Guide
  - How to Transfer a Reservation to Another Person
  - What to Do If an Experience Is Sold Out
  - CultPass Referral Program
  - Contacting Human Support and Escalation Policy


In [14]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    print(f"Users: {len(users)}")
    for user in users:
        print(f"  {user}")

Users: 1
  <User(user_id='74d6e139-2ace-4df3-abb4-cbd82d90b059', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [15]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    ticket = user.tickets[0]
    print(f"Ticket: {ticket}")
    print(f"Metadata: {ticket.ticket_metadata}")
    for message in ticket.messages:
        print(f"  Message: {message}")

Ticket: <Ticket(ticket_id='92139507-8ca4-451c-874f-e5b3d3c50a07', channel='chat', created_at='2026-07-03 01:16:21')>
Metadata: <TicketMetadata(ticket_id='92139507-8ca4-451c-874f-e5b3d3c50a07', status='open', issue_type='None')>
  Message: <TicketMessage(message_id='c46f2522-f417-45ea-aed3-f6127b6fc445', role='user', content='I can't log in to my Cultpass ...')>
